# AMR-HUB behaviour-cloning triage

Notebook version of `bc-prototype.py`: database connection, schema checks, room resolution and workstation inference, shift construction, learnability tests. Expensive stages run only when their `RUN_*` flag is set. Outputs go to `/outputs`; the database is read-only under `/data`.

## 0. Configuration

The database is found automatically if `/data` contains one `.db`/`.duckdb` file.

In [ ]:
from __future__ import annotations

import contextlib
import csv
import io
import math
import os
import re
import traceback
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

DATA_DIR = Path(os.environ.get("AMR_DATA_DIR", "/data"))
OUTPUT_DIR = Path(os.environ.get("AMR_OUTPUT_DIR", "/outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional explicit database path.
DB_PATH: Path | None = None

CROSSWALK_PATH: Path | None = (
    (OUTPUT_DIR / "ws_crosswalk.csv")
    if (OUTPUT_DIR / "ws_crosswalk.csv").exists()
    else None
)
NURSING_PATH: Path | None = None

# Expensive stages.
RUN_INFER_WS = False
RUN_LOAD_SHIFTS = False
RUN_BATTERY = False
RUN_RULEOUT = False
RUN_SWEEP = False

ACTION_NAME = "next_loc"  # next_loc, next_class, next_itype
MARKOV_ORDER = 1
USE_NEURAL_BC = False  # JAX/Flax/Optax not installed in the image

# Row limit for quick checks only; truncates shifts.
EVENT_ROW_LIMIT: int | None = None

## 1. Open the database

In [ ]:
def discover_database(data_dir: Path) -> Path:
    candidates = sorted({*data_dir.glob("*.db"), *data_dir.glob("*.duckdb")})
    if not candidates:
        raise FileNotFoundError(
            f"No .db or .duckdb file found in {data_dir}. "
            "Mount the database directory at /data or set DB_PATH explicitly."
        )
    if len(candidates) > 1:
        joined = "\n  ".join(str(p) for p in candidates)
        raise RuntimeError(
            "Multiple database files found; set DB_PATH explicitly:\n  " + joined
        )
    return candidates[0]


if DB_PATH is None:
    DB_PATH = discover_database(DATA_DIR)
DB_PATH = DB_PATH.expanduser().resolve()
if not DB_PATH.is_file():
    raise FileNotFoundError(DB_PATH)

print("database:", DB_PATH)
print("size GiB:", round(DB_PATH.stat().st_size / (1024**3), 3))
print("duckdb:", duckdb.__version__)

con = duckdb.connect(str(DB_PATH), read_only=True)
print("connection: read-only OK")

## 2. Catalog inventory

In [ ]:
def query_df(
    sql: str, params: list[Any] | None = None, *, label: str | None = None
) -> pd.DataFrame:
    try:
        df = con.execute(sql, params or []).df()
        if label:
            print(f"[{label}] {len(df):,} rows")
        return df
    except Exception as exc:
        print(
            f"QUERY FAILED{f' [{label}]' if label else ''}: {type(exc).__name__}: {exc}"
        )
        print("SQL:\n", sql)
        raise


catalog_df = query_df(
    """
    select table_catalog, table_schema, table_name, table_type
    from information_schema.tables
    where table_schema not in ('information_schema', 'pg_catalog')
    order by table_schema, table_name
""",
    label="catalog",
)
display(catalog_df)

## 3. Schema check

Missing tables/columns, and whether `Ref.Staff` has `Role` or only `StaffGroup`.

In [ ]:
EXPECTED_COLUMNS = {
    "Data.StaffLocationEvent": [
        "MasterIndexId",
        "EventDateTime",
        "SourceKey",
        "LinkKey",
        "PatientDurableKey",
        "InteractionTypeKey",
    ],
    "Ref.InteractionType": [
        "InteractionTypeKey",
        "InteractionType",
        "InteractionTypeClass",
    ],
    "Ref.Staff": ["MasterIndexId"],
    "Ref.Door": ["DoorKey", "DoorName"],
    "Ref.Workstation": ["WorkstationKey", "WorkstationName"],
    "Ref.Roster": ["RosterKey", "RosterName"],
    "Ref.Department": ["DepartmentKey", "BedName", "RoomName"],
}


def split_table_name(qualified: str) -> tuple[str, str]:
    if "." not in qualified:
        return "main", qualified
    return tuple(qualified.split(".", 1))  # type: ignore[return-value]


def table_columns(qualified: str) -> pd.DataFrame:
    schema, table = split_table_name(qualified)
    return con.execute(
        """
        select column_name, data_type, is_nullable, ordinal_position
        from information_schema.columns
        where lower(table_schema) = lower(?) and lower(table_name) = lower(?)
        order by ordinal_position
    """,
        [schema, table],
    ).df()


def schema_preflight() -> pd.DataFrame:
    records = []
    for table, expected in EXPECTED_COLUMNS.items():
        cols_df = table_columns(table)
        actual = {
            c.lower(): c
            for c in cols_df.get("column_name", pd.Series(dtype=str)).tolist()
        }
        if cols_df.empty:
            records.append({"table": table, "status": "MISSING TABLE", "detail": ""})
            continue
        missing = [c for c in expected if c.lower() not in actual]
        records.append(
            {
                "table": table,
                "status": "OK" if not missing else "MISSING COLUMNS",
                "detail": ", ".join(missing),
            }
        )
    return pd.DataFrame(records)


preflight_df = schema_preflight()
display(preflight_df)

staff_cols = table_columns("Ref.Staff")
display(staff_cols)
staff_col_names = {str(c).lower(): str(c) for c in staff_cols.get("column_name", [])}
ROLE_COLUMN = staff_col_names.get("role") or staff_col_names.get("staffgroup")
print("role column selected:", ROLE_COLUMN or "none (will use 'unknown')")

## 4. Probe queries

Timestamp expression, event/interaction join, reference-key coverage, roster start/end pairing.

In [ ]:
EVENT_TIME_EXPR = "epoch(e.EventDateTime)"

role_expr = (
    f"coalesce(cast(s.{ROLE_COLUMN} as varchar), 'unknown')"
    if ROLE_COLUMN
    else "'unknown'"
)
patient_expr = (
    "cast(e.PatientDurableKey as varchar)"
    if "patientdurablekey"
    in {
        str(c).lower()
        for c in table_columns("Data.StaffLocationEvent").get("column_name", [])
    }
    else "cast(NULL as varchar)"
)

probe_sql = f"""
select
    cast(e.MasterIndexId as varchar) as staff,
    {EVENT_TIME_EXPR} as ts,
    e.EventDateTime,
    e.SourceKey,
    e.LinkKey,
    {patient_expr} as patient,
    it.InteractionType as itype,
    it.InteractionTypeClass as iclass,
    {role_expr} as role
from Data.StaffLocationEvent e
join Ref.InteractionType it
  on e.InteractionTypeKey = it.InteractionTypeKey
left join Ref.Staff s
  on s.MasterIndexId = e.MasterIndexId
limit 20
"""
display(query_df(probe_sql, label="event sample"))

In [ ]:
diagnostic_queries = {
    "event volume and timestamp range": """
        select count(*) as n_events,
               min(EventDateTime) as min_time,
               max(EventDateTime) as max_time,
               count(distinct MasterIndexId) as n_staff
        from Data.StaffLocationEvent
    """,
    "interaction classes": """
        select it.InteractionTypeClass, it.InteractionType, count(*) as n
        from Data.StaffLocationEvent e
        join Ref.InteractionType it using (InteractionTypeKey)
        group by 1, 2 order by 1, n desc
    """,
    "unmatched interaction keys": """
        select count(*) as unmatched
        from Data.StaffLocationEvent e
        left join Ref.InteractionType it using (InteractionTypeKey)
        where it.InteractionTypeKey is null
    """,
    "unmatched staff IDs": """
        select count(*) as unmatched_events,
               count(distinct e.MasterIndexId) as unmatched_staff
        from Data.StaffLocationEvent e
        left join Ref.Staff s using (MasterIndexId)
        where s.MasterIndexId is null
    """,
    "flowsheet source-key coverage in department": """
        select
          count(*) as flowsheet_events,
          count(d.DepartmentKey) as matched_events,
          count(distinct e.SourceKey) as distinct_source_keys,
          count(distinct d.DepartmentKey) as matched_source_keys
        from Data.StaffLocationEvent e
        join Ref.InteractionType it using (InteractionTypeKey)
        left join Ref.Department d on d.DepartmentKey = e.SourceKey
        where it.InteractionTypeClass = 'Flowsheet'
    """,
    "roster event summary": """
        select it.InteractionType, count(*) as n,
               count(e.LinkKey) as with_link_key,
               count(e.SourceKey) as with_source_key
        from Data.StaffLocationEvent e
        join Ref.InteractionType it using (InteractionTypeKey)
        where it.InteractionTypeClass = 'Roster'
        group by 1 order by 1
    """,
    "roster pair quality": """
        with roster as (
          select cast(e.MasterIndexId as varchar) staff,
                 e.LinkKey,
                 it.InteractionType,
                 e.EventDateTime,
                 row_number() over (
                    partition by e.MasterIndexId, e.LinkKey, it.InteractionType
                    order by e.EventDateTime
                 ) as occurrence
          from Data.StaffLocationEvent e
          join Ref.InteractionType it using (InteractionTypeKey)
          where it.InteractionTypeClass = 'Roster' and e.LinkKey is not null
        ), starts as (
          select staff, LinkKey, occurrence, EventDateTime as start_time
          from roster where InteractionType = 'Roster Start'
        ), ends as (
          select staff, LinkKey, occurrence, EventDateTime as end_time
          from roster where InteractionType = 'Roster End'
        )
        select
          count(*) filter (where s.start_time is not null) as starts,
          count(*) filter (where e.end_time is not null) as ends,
          count(*) filter (where s.start_time is not null and e.end_time is not null) as paired,
          count(*) filter (where e.end_time <= s.start_time) as non_monotonic
        from starts s
        full join ends e using (staff, LinkKey, occurrence)
    """,
}

for label, sql in diagnostic_queries.items():
    print("\n###", label)
    try:
        display(query_df(sql))
    except Exception:
        print("Continue to the next diagnostic after inspecting the traceback above.")

## 5. Model and spatial settings

In [ ]:
N_TIME_BUCKETS = 4  # 0-6, 6-12, 12-18, 18-24
KEEP_UNROSTERED = False  # events outside any rostered interval
MIN_SHIFT_EVENTS = 3  # shifts shorter than this are dropped
TOPK = 3

# Prefixes for unresolved keys, so keys from different classes do not collide.
CLASS_PREFIX = {
    "Door Message": "door",
    "Workstation": "ws",
    "Flowsheet": "fs",
    "Roster": "roster",
}

# Reference table giving a fallback name for each class's SourceKey.
SOURCE_REF = {
    "Door Message": ("Ref.Door", "DoorKey", "DoorName"),
    "Workstation": ("Ref.Workstation", "WorkstationKey", "WorkstationName"),
    "Roster": ("Ref.Roster", "RosterKey", "RosterName"),
}
FLOWSHEET_SOURCE_TABLE = None  # e.g. ("Ref.Department", "DepartmentKey", "RoomName")

# Shared room frame. Priority per key: crosswalk CSV > nursing CSV > Ref.Department
# join > door-name edge parse > namespaced fallback. Doors resolve to an edge
# "door:A|B" because the side of the door is unknown.
ROOM_RESOLUTION: dict[str, tuple[str, str, str] | None] = {
    "Door Message": None,  # edge-parsed from DoorName instead
    "Workstation": None,  # inferred via `infer-ws` instead
    "Flowsheet": ("Ref.Department", "DepartmentKey", "RoomName"),  # fallback tier
}

# No Ref.Room table in this database.
ROOM_NAME_TABLE: tuple[str, str, str] | None = None

# Crosswalk CSV (iclass,source_key,room); blank rooms are skipped.
CROSSWALK_CSV: str | None = None

# Nursing table CSV: source_key,room or bed,room (bed joined via Ref.Department).
NURSING_CSV: str | None = None

DOOR_EDGES = True
DOOR_NAME_SEPARATORS = r"\s+(?:-|\u2013|\u2014|<->|/|to)\s+"


def std_code(s: str) -> str:
    """Trim, collapse whitespace and upper-case a location code."""
    return re.sub(r"\s+", " ", s.strip()).upper()


# Workstation inference: modal flowsheet room within +/- WS_WINDOW_MIN minutes.
WS_WINDOW_MIN = 10
WS_MIN_SUPPORT = 20
WS_MIN_SHARE = 0.6

## 6. Spatial resolution and shift building

In [ ]:
@dataclass
class Step:
    ts: float  # epoch seconds
    location: str  # canonical "room:<code>" / "door:A|B" (or fallback "ws:...")
    iclass: str  # InteractionTypeClass
    itype: str  # InteractionType (fine)
    role: str
    patient: str | None
    time_bucket: int
    unit: str = "unknown"  # rostered unit (Ref.Roster), shift-level


@dataclass
class Shift:
    staff: str
    link_key: int
    unit: str = "unknown"
    steps: list[Step] = field(default_factory=list)


def _load_ref_map(con, table, key_col, name_col) -> dict[int, str]:
    rows = con.execute(f"select {key_col}, {name_col} from {table}").fetchall()
    return {int(k): str(v) for k, v in rows if k is not None}


def _norm_name(s: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


def _load_nursing_map(con, nursing_csv: str) -> dict[int, str]:
    """Nursing-staff table -> {Flowsheet SourceKey: 'room:<code>'}.

    Direct layout (source_key column) is used as-is. Bed-name layout is
    joined through Ref.Department on a normalised match against BedName,
    then RoomName, to recover DepartmentKey (= SourceKey)."""
    with open(nursing_csv, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        cols = {c.lower().strip(): c for c in (reader.fieldnames or [])}
        rows = list(reader)

    def col(*names):
        for n in names:
            if n in cols:
                return cols[n]
        return None

    key_c = col("source_key", "sourcekey", "key", "departmentkey")
    bed_c = col("bed", "bed_name", "bedname", "bed_location", "location")
    room_c = col("room", "room_code", "roomcode")
    if room_c is None or (key_c is None and bed_c is None):
        raise SystemExit(
            f"[nursing] {nursing_csv}: need a room column plus either a "
            f"source_key or bed-name column; found {list(cols)}"
        )

    out: dict[int, str] = {}
    if key_c is not None:
        for r in rows:
            if r[key_c] and r[room_c] and r[room_c].strip():
                out[int(r[key_c])] = f"room:{std_code(r[room_c])}"
        print(f"[nursing] {len(out)} flowsheet keys mapped directly.")
        return out

    dep = con.execute(
        "select DepartmentKey, BedName, RoomName from Ref.Department"
    ).fetchall()
    by_bed: dict[str, list[int]] = defaultdict(list)
    by_room: dict[str, list[int]] = defaultdict(list)
    for k, b, rn in dep:
        if k is None:
            continue
        if b:
            by_bed[_norm_name(b)].append(int(k))
        if rn:
            by_room[_norm_name(rn)].append(int(k))

    unmatched = []
    for r in rows:
        bed, room = r.get(bed_c, ""), r.get(room_c, "")
        if not bed or not room or not room.strip():
            continue
        keys = by_bed.get(_norm_name(bed)) or by_room.get(_norm_name(bed))
        if not keys:
            unmatched.append(bed)
            continue
        for k in keys:
            out[k] = f"room:{std_code(room)}"
    print(
        f"[nursing] {len(out)} flowsheet keys mapped via bed names; "
        f"{len(unmatched)} nursing rows unmatched"
        + (f" (e.g. {unmatched[:3]!r})" if unmatched else "")
        + "."
    )
    return out


def _parse_door_edges(door_names: dict[int, str]) -> dict[int, str]:
    """DoorName 'A <sep> B' -> {DoorKey: 'door:A|B'} with standardised,
    order-independent endpoints. Names that do not split into exactly two
    parts are left out (they fall back to the raw namespaced name)."""
    out: dict[int, str] = {}
    for k, name in door_names.items():
        parts = [std_code(p) for p in re.split(DOOR_NAME_SEPARATORS, name) if p.strip()]
        if len(parts) == 2:
            a, b = sorted(parts)
            out[k] = f"door:{a}|{b}"
    return out


def _load_room_maps(
    con, crosswalk_csv: str | None, nursing_csv: str | None = None
) -> dict[str, dict[int, str]]:
    """Per-class {source_key -> canonical symbol}: 'room:<code>' for points,
    'door:A|B' for door edges.

    Priority (later overrides earlier): ROOM_RESOLUTION joins -> door edge
    parse -> nursing CSV -> crosswalk CSV. Any class with no mapping at all
    is absent from the dict and uses the namespaced fallback."""
    room_names: dict[int, str] = {}
    if ROOM_NAME_TABLE is not None:
        try:
            room_names = _load_ref_map(con, *ROOM_NAME_TABLE)
        except duckdb.Error:
            print(
                f"[rooms] {ROOM_NAME_TABLE[0]} not readable; using raw room keys as symbols."
            )

    def room_symbol(room_val) -> str:
        try:
            k = int(room_val)
            name = room_names.get(k)
            return f"room:{std_code(name)}" if name is not None else f"room:{k}"
        except (TypeError, ValueError):
            # Same standardisation as door endpoints so spellings match.
            return f"room:{std_code(str(room_val))}"

    maps: dict[str, dict[int, str]] = {}
    for iclass, spec in ROOM_RESOLUTION.items():
        if spec is None:
            continue
        tbl, kcol, rcol = spec
        try:
            rows = con.execute(f"select {kcol}, {rcol} from {tbl}").fetchall()
        except duckdb.Error as e:
            print(
                f"[rooms] {iclass}: cannot read {rcol} from {tbl} "
                f"({type(e).__name__}); leaving this class on the namespaced fallback."
            )
            continue
        maps[iclass] = {
            int(k): room_symbol(r) for k, r in rows if k is not None and r is not None
        }

    if DOOR_EDGES:
        try:
            door_names = _load_ref_map(con, *SOURCE_REF["Door Message"])
            edges = _parse_door_edges(door_names)
            maps.setdefault("Door Message", {}).update(edges)
            print(
                f"[rooms] door edges parsed: {len(edges)}/{len(door_names)} "
                "door names split into two endpoints."
            )
        except duckdb.Error:
            pass

    if nursing_csv:
        maps.setdefault("Flowsheet", {}).update(_load_nursing_map(con, nursing_csv))

    if crosswalk_csv:
        n_rows = n_skip = 0
        with open(crosswalk_csv, newline="", encoding="utf-8-sig") as f:
            for row in csv.DictReader(f):
                room = (row.get("room") or "").strip()
                if not room:
                    n_skip += 1
                    continue
                iclass = row["iclass"].strip()
                key = int(row["source_key"])
                maps.setdefault(iclass, {})[key] = f"room:{std_code(room)}"
                n_rows += 1
        print(
            f"[rooms] crosswalk {crosswalk_csv}: {n_rows} mappings applied"
            + (f", {n_skip} blank rows skipped" if n_skip else "")
            + "."
        )

    return maps


def load_shifts(
    con, crosswalk_csv: str | None = None, nursing_csv: str | None = None
) -> tuple[list[Shift], Counter]:
    """Build per-(staff, shift) touchpoint sequences from DuckDB.

    Returns (shifts, resolution_stats) where resolution_stats counts, per
    (iclass, outcome), how each event's location was resolved:
    outcome in {"room", "edge", "fallback", "null"}.
    """
    loc_maps: dict[str, dict[int, str]] = {}
    for iclass, (tbl, kcol, ncol) in SOURCE_REF.items():
        loc_maps[iclass] = _load_ref_map(con, tbl, kcol, ncol)
    if FLOWSHEET_SOURCE_TABLE:
        tbl, kcol, ncol = FLOWSHEET_SOURCE_TABLE
        loc_maps["Flowsheet"] = _load_ref_map(con, tbl, kcol, ncol)

    room_maps = _load_room_maps(con, crosswalk_csv, nursing_csv)
    roster_names = loc_maps.get("Roster", {})
    res_stats: Counter = Counter()

    def resolve(iclass: str, source_key) -> str:
        prefix = CLASS_PREFIX.get(iclass, "x")
        if source_key is None:
            res_stats[(iclass, "null")] += 1
            return f"{prefix}:na"
        k = int(source_key)
        sym = room_maps.get(iclass, {}).get(k)
        if sym is not None:
            res_stats[(iclass, "room" if sym.startswith("room:") else "edge")] += 1
            return sym
        res_stats[(iclass, "fallback")] += 1
        name = loc_maps.get(iclass, {}).get(k)
        return f"{prefix}:{name if name is not None else source_key}"

    q = """
    select
        e.MasterIndexId          as staff,
        epoch(e.EventDateTime)   as ts,
        e.SourceKey              as source_key,
        e.LinkKey                as link_key,
        e.PatientDurableKey      as patient,
        it.InteractionType       as itype,
        it.InteractionTypeClass  as iclass,
        coalesce(s.Role, 'unknown')       as role
    from Data.StaffLocationEvent e
    join Ref.InteractionType it
      on e.InteractionTypeKey = it.InteractionTypeKey
    left join Ref.Staff s
      on s.MasterIndexId = e.MasterIndexId
    order by e.MasterIndexId, e.EventDateTime
    """
    rows = con.execute(q).fetchall()

    # Roster intervals per staff by LinkKey; Roster Start's SourceKey is the unit.
    starts: dict[tuple[str, int], tuple[float, str]] = {}
    ends: dict[tuple[str, int], float] = {}
    for staff, ts, sk, link_key, patient, itype, iclass, role in rows:
        if iclass != "Roster" or link_key is None:
            continue
        key = (staff, int(link_key))
        if itype == "Roster Start":
            unit = roster_names.get(int(sk), str(sk)) if sk is not None else "unknown"
            starts[key] = (float(ts), unit)
        elif itype == "Roster End":
            ends[key] = float(ts)

    intervals: dict[str, list[tuple[float, float, int, str]]] = defaultdict(list)
    for key, (t0, unit) in starts.items():
        t1 = ends.get(key)
        if t1 is None or t1 <= t0:
            continue  # unmatched or non-monotonic pair -> skip (report separately)
        staff, link_key = key
        intervals[staff].append((t0, t1, link_key, unit))
    for staff in intervals:
        intervals[staff].sort()

    def which_shift(staff: str, ts: float) -> tuple[int, str] | None:
        for t0, t1, link_key, unit in intervals.get(
            staff, ()
        ):  # linear; fine for a prototype
            if t0 <= ts <= t1:
                return link_key, unit
        return None

    shifts: dict[tuple[str, int], Shift] = {}
    for staff, ts, sk, link_key, patient, itype, iclass, role in rows:
        if iclass == "Roster":
            continue
        ts = float(ts)
        hit = which_shift(staff, ts)
        if hit is None:
            if not KEEP_UNROSTERED:
                continue
            sh, unit = -1, "unknown"
        else:
            sh, unit = hit
        skey = (staff, sh)
        if skey not in shifts:
            shifts[skey] = Shift(staff=staff, link_key=sh, unit=unit)
        tb = min(N_TIME_BUCKETS - 1, int(((ts % 86400) / 86400) * N_TIME_BUCKETS))
        shifts[skey].steps.append(
            Step(
                ts=ts,
                location=resolve(iclass, sk),
                iclass=iclass,
                itype=itype,
                role=str(role),
                patient=(str(patient) if patient is not None else None),
                time_bucket=tb,
                unit=unit,
            )
        )

    kept = [shift for shift in shifts.values() if len(shift.steps) >= MIN_SHIFT_EVENTS]
    for shift in kept:
        shift.steps.sort(key=lambda st: st.ts)
    return kept, res_stats


def report_spatial_frame(shifts: list[Shift], res_stats: Counter) -> None:
    """Room-frame coverage per class, and whether rooms are shared across classes."""
    print("\n--- shared spatial frame ---")
    for iclass in sorted({c for c, _ in res_stats}):
        room = res_stats.get((iclass, "room"), 0)
        edge = res_stats.get((iclass, "edge"), 0)
        fb = res_stats.get((iclass, "fallback"), 0)
        nul = res_stats.get((iclass, "null"), 0)
        tot = max(1, room + edge + fb + nul)
        print(
            f"  {iclass:14s} room {room / tot:6.1%}   edge {edge / tot:6.1%}   "
            f"fallback {fb / tot:6.1%}   null {nul / tot:6.1%}   ({tot} events)"
        )

    room_classes: dict[str, set[str]] = defaultdict(set)
    endpoints: set[str] = set()
    for sh in shifts:
        for s in sh.steps:
            if s.location.startswith("room:"):
                room_classes[s.location].add(s.iclass)
            elif s.location.startswith("door:") and "|" in s.location:
                endpoints.update(s.location[len("door:") :].split("|"))
    multi = sum(1 for v in room_classes.values() if len(v) >= 2)
    print(
        f"  rooms in shared frame: {len(room_classes)}; containing >=2 classes: {multi}"
    )
    room_capable = {c for (c, outcome) in res_stats if outcome == "room"}
    if len(room_capable) >= 2 and multi == 0:
        print(
            "  WARNING: no room contains more than one class; check spellings / crosswalk."
        )
    elif len(room_capable) < 2 and room_classes:
        print("  (only one class resolves to rooms so far)")

    # If no door endpoint matches a room, spellings differ; extend std_code().
    if endpoints:
        rooms = {r[len("room:") :] for r in room_classes}
        matched = sorted(endpoints & rooms)
        unmatched = sorted(endpoints - rooms)
        print(
            f"  door endpoints: {len(endpoints)} distinct; "
            f"{len(matched)} match a room code"
            + (f" (e.g. {matched[:3]})" if matched else "")
            + "."
        )
        if unmatched:
            print(f"  endpoints with no matching room: {unmatched}")
        if not matched:
            print("  no endpoint matches a room name; check std_code().")

### Reference maps and room resolution

In [ ]:
for table, key_col, name_col in [
    ("Ref.Door", "DoorKey", "DoorName"),
    ("Ref.Workstation", "WorkstationKey", "WorkstationName"),
    ("Ref.Roster", "RosterKey", "RosterName"),
]:
    print(f"\n{table}")
    try:
        display(query_df(f"select {key_col}, {name_col} from {table} order by 1"))
    except Exception:
        pass

try:
    room_maps_preview = _load_room_maps(
        con,
        str(CROSSWALK_PATH) if CROSSWALK_PATH else None,
        str(NURSING_PATH) if NURSING_PATH else None,
    )
    print("\nresolved key counts:", {k: len(v) for k, v in room_maps_preview.items()})
    for iclass, mapping in room_maps_preview.items():
        print(f"{iclass}: first mappings", list(mapping.items())[:10])
except Exception:
    traceback.print_exc()

## 7. Workstation locations

Requires `RUN_INFER_WS = True`. Writes `/outputs/ws_crosswalk.csv`.

In [ ]:
def run_infer_ws(
    con,
    nursing_csv: str | None,
    out_path: str,
    window_min: int = WS_WINDOW_MIN,
    min_support: int = WS_MIN_SUPPORT,
    min_share: float = WS_MIN_SHARE,
) -> None:
    """Assign each workstation the modal flowsheet room charted by the same staff
    member within +/- window_min minutes, if support and share pass the thresholds.
    Other rows are written with a blank room. Mobile workstations may have no
    fixed room, so a low share can mean the workstation moves."""
    room_maps = _load_room_maps(con, None, nursing_csv)
    fs_rooms = room_maps.get("Flowsheet", {})
    if not fs_rooms:
        raise SystemExit(
            "[infer-ws] no flowsheet room mapping available; "
            "provide --nursing or fix ROOM_RESOLUTION first."
        )
    ws_names = _load_ref_map(con, *SOURCE_REF["Workstation"])

    rows = con.execute("""
        select e.MasterIndexId, epoch(e.EventDateTime), e.SourceKey,
               it.InteractionTypeClass
        from Data.StaffLocationEvent e
        join Ref.InteractionType it
          on e.InteractionTypeKey = it.InteractionTypeKey
        where it.InteractionTypeClass in ('Workstation', 'Flowsheet')
          and e.SourceKey is not null
        order by e.MasterIndexId, e.EventDateTime
    """).fetchall()

    window = window_min * 60.0
    votes: dict[int, Counter] = defaultdict(Counter)
    from itertools import groupby

    # Two-pointer sweep of each staff member's flowsheet events.
    for _staff, grp in groupby(rows, key=lambda r: r[0]):
        evs = [(float(ts), int(sk), ic) for _s, ts, sk, ic in grp]
        fs = [
            (ts, fs_rooms[sk])
            for ts, sk, ic in evs
            if ic == "Flowsheet" and sk in fs_rooms
        ]
        if not fs:
            continue
        lo = 0
        for ts, sk, ic in evs:
            if ic != "Workstation":
                continue
            while lo < len(fs) and fs[lo][0] < ts - window:
                lo += 1
            j = lo
            while j < len(fs) and fs[j][0] <= ts + window:
                votes[sk][fs[j][1]] += 1
                j += 1

    n_conf = 0
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(
            [
                "iclass",
                "source_key",
                "name_hint",
                "room",
                "support",
                "share",
                "runner_up",
            ]
        )
        for sk in sorted(ws_names):
            ctr = votes.get(sk, Counter())
            support = sum(ctr.values())
            if support == 0:
                w.writerow(["Workstation", sk, ws_names[sk], "", 0, "", ""])
                continue
            ranked = ctr.most_common(2)
            best_room, best_n = ranked[0]
            runner = ranked[1][0].removeprefix("room:") if len(ranked) > 1 else ""
            share = best_n / support
            confident = support >= min_support and share >= min_share
            room = best_room.removeprefix("room:") if confident else ""
            if confident:
                n_conf += 1
            w.writerow(
                ["Workstation", sk, ws_names[sk], room, support, f"{share:.2f}", runner]
            )
    print(
        f"[infer-ws] {n_conf}/{len(ws_names)} workstations placed "
        f"(window=+/-{window_min}min, support>={min_support}, share>={min_share})."
    )
    print(f"[infer-ws] wrote {out_path}; low-share rows left blank.")

In [ ]:
WS_CROSSWALK_OUT = OUTPUT_DIR / "ws_crosswalk.csv"

if RUN_INFER_WS:
    run_infer_ws(
        con,
        nursing_csv=str(NURSING_PATH) if NURSING_PATH else None,
        out_path=str(WS_CROSSWALK_OUT),
    )
    CROSSWALK_PATH = WS_CROSSWALK_OUT
    print("crosswalk ready:", CROSSWALK_PATH)
    display(pd.read_csv(CROSSWALK_PATH).head(50))
else:
    print("Skipped. Set RUN_INFER_WS = True to execute this stage.")

## 8. Build rostered shifts

Requires `RUN_LOAD_SHIFTS = True`. Loads the full event stream.

In [ ]:
shifts: list[Shift] | None = None
res_stats: Counter | None = None

if RUN_LOAD_SHIFTS:
    try:
        shifts, res_stats = load_shifts(
            con,
            crosswalk_csv=str(CROSSWALK_PATH) if CROSSWALK_PATH else None,
            nursing_csv=str(NURSING_PATH) if NURSING_PATH else None,
        )
        print(
            f"loaded {len(shifts):,} shifts, "
            f"{sum(len(s.steps) for s in shifts):,} touchpoints, "
            f"{len({s.staff for s in shifts}):,} staff"
        )
        report_spatial_frame(shifts, res_stats)
    except Exception:
        traceback.print_exc()
        raise
else:
    print("Skipped. Set RUN_LOAD_SHIFTS = True after the schema probes pass.")

### Shift checks

In [ ]:
if shifts:
    shift_summary = pd.DataFrame(
        [
            {
                "staff": sh.staff,
                "link_key": sh.link_key,
                "unit": sh.unit,
                "n_steps": len(sh.steps),
                "start_ts": sh.steps[0].ts,
                "end_ts": sh.steps[-1].ts,
                "ordered": all(a.ts <= b.ts for a, b in zip(sh.steps, sh.steps[1:])),
            }
            for sh in shifts[:1000]
        ]
    )
    display(shift_summary.head(20))

    sample_steps = pd.DataFrame(
        [
            vars(step) | {"staff": sh.staff, "link_key": sh.link_key}
            for sh in shifts[:3]
            for step in sh.steps[:30]
        ]
    )
    display(sample_steps)
else:
    print("No shifts loaded yet.")

## 9. State/action and learnability functions

In [ ]:
STATE_VARIANTS = {
    "S1_loc": lambda s, prev: (s.location,),
    "S2_loc_time": lambda s, prev: (s.location, s.time_bucket),
    "S3_loc_time_role": lambda s, prev: (s.location, s.time_bucket, s.role),
    "S4_loc_time_role_class": lambda s, prev: (
        s.location,
        s.time_bucket,
        s.role,
        s.iclass,
    ),
    "S5_bigram": lambda s, prev: (
        s.location,
        s.time_bucket,
        s.role,
        (prev.location if prev else "<s>"),
    ),
    "S6_bigram_unit": lambda s, prev: (
        s.location,
        s.time_bucket,
        s.role,
        (prev.location if prev else "<s>"),
        s.unit,
    ),
}
ACTION_KEYS = {
    "next_loc": lambda s: s.location,
    "next_class": lambda s: s.iclass,
    "next_itype": lambda s: s.itype,
}


def build_pairs(shifts, state_fn, action_key):
    """Return list of (state_tuple, action_str) across all shifts."""
    pairs = []
    for sh in shifts:
        prev = None
        for cur, nxt in zip(sh.steps, sh.steps[1:]):
            pairs.append((state_fn(cur, prev), action_key(nxt)))
            prev = cur
    return pairs


def split_pairs(pairs, frac=0.8, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(pairs))
    cut = int(len(pairs) * frac)
    tr = [pairs[i] for i in idx[:cut]]
    te = [pairs[i] for i in idx[cut:]]
    return tr, te

In [ ]:
def conditional_entropy(pairs) -> tuple[float, float]:
    """H(action) and H(action | state), in bits. Gap = mutual information."""
    a_counts = Counter(a for _, a in pairs)
    n = sum(a_counts.values())
    H_a = -sum((c / n) * math.log2(c / n) for c in a_counts.values())

    by_state: defaultdict[tuple, Counter] = defaultdict(Counter)
    for s, a in pairs:
        by_state[s][a] += 1
    H_a_given_s = 0.0
    for s, ctr in by_state.items():
        m = sum(ctr.values())
        h = -sum((c / m) * math.log2(c / m) for c in ctr.values())
        H_a_given_s += (m / n) * h
    return H_a, H_a_given_s


def variant_stats(shifts, action_key) -> dict:
    """Distinct action paths across shifts and share of the most common one."""
    paths = Counter(tuple(action_key(s) for s in sh.steps) for sh in shifts)
    total = sum(paths.values())
    top = paths.most_common(1)[0][1] if paths else 0
    return {
        "n_shifts": len(shifts),
        "distinct_paths": len(paths),
        "top_path_share": round(top / total, 3) if total else 0.0,
        "singletons": sum(1 for c in paths.values() if c == 1),
    }


class MarkovModel:
    """Order-k categorical model P(action | last-k actions), Laplace-smoothed."""

    def __init__(self, order=1, alpha=0.5):
        self.order = order
        self.alpha = alpha
        self.table: dict[tuple, Counter] = defaultdict(Counter)
        self.vocab: set = set()

    def fit(self, seq_pairs):
        for ctx, a in seq_pairs:
            self.table[ctx][a] += 1
            self.vocab.add(a)

    def _dist(self, ctx) -> dict:
        ctr = self.table.get(ctx, Counter())
        V = max(1, len(self.vocab))
        denom = sum(ctr.values()) + self.alpha * V
        return {a: (ctr.get(a, 0) + self.alpha) / denom for a in self.vocab}

    def evaluate(self, seq_pairs, topk=TOPK):
        top1 = topk_hit = 0
        ll = 0.0
        for ctx, a in seq_pairs:
            dist = self._dist(ctx)
            ranked = sorted(dist, key=dist.get, reverse=True)
            if ranked and ranked[0] == a:
                top1 += 1
            if a in ranked[:topk]:
                topk_hit += 1
            p = dist.get(a, self.alpha / max(1, len(self.vocab)))
            ll += math.log2(max(p, 1e-12))
        n = max(1, len(seq_pairs))
        return {
            "top1": round(top1 / n, 3),
            f"top{topk}": round(topk_hit / n, 3),
            "perplexity": round(2 ** (-ll / n), 2),
        }


def run_battery(shifts, action_name="next_loc", order=1):
    action_key = ACTION_KEYS[action_name]
    print(f"\n=== Learnability battery (action = {action_name}) ===")
    print("variants:", variant_stats(shifts, action_key))

    # Markov context = last `order` actions.
    seq = []
    for sh in shifts:
        acts = [action_key(s) for s in sh.steps]
        for i in range(order, len(acts)):
            ctx = tuple(acts[i - order : i])
            seq.append((ctx, acts[i]))
    tr, te = split_pairs(seq)
    m = MarkovModel(order=order)
    m.fit(tr)
    print(f"markov(order={order}) held-out:", m.evaluate(te))

    pairs = build_pairs(shifts, STATE_VARIANTS["S5_bigram"], action_key)
    H_a, H_ags = conditional_entropy(pairs)
    print(
        f"entropy: H(a)={H_a:.2f} bits  H(a|state)={H_ags:.2f} bits  "
        f"info gain={H_a - H_ags:.2f} bits"
    )

In [ ]:
def grouped_split(shifts, frac=0.8, seed=0):
    """Split by shift, so all pairs from a shift stay on one side."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(shifts))
    cut = int(len(shifts) * frac)
    return [shifts[i] for i in idx[:cut]], [shifts[i] for i in idx[cut:]]


def _topk_eval(predict, test_pairs, topk=TOPK):
    """predict: state_tuple -> ranked list of actions (best first)."""
    top1 = topk_hit = 0
    for s, a in test_pairs:
        ranked = predict(s)
        if ranked and ranked[0] == a:
            top1 += 1
        if a in ranked[:topk]:
            topk_hit += 1
    n = max(1, len(test_pairs))
    return round(top1 / n, 3), round(topk_hit / n, 3)


def run_ruleout(shifts, action_name="next_loc", use_bc=True):
    action_key = ACTION_KEYS[action_name]
    tr_shifts, te_shifts = grouped_split(shifts)
    print(f"\n=== BC rule-out (action = {action_name}) ===")
    print(f"grouped split: {len(tr_shifts)} train shifts, {len(te_shifts)} test shifts")

    # Majority class.
    tr_actions = [action_key(s) for sh in tr_shifts for s in sh.steps]
    marginal_rank = [a for a, _ in Counter(tr_actions).most_common()]
    test_min = build_pairs(te_shifts, STATE_VARIANTS["S1_loc"], action_key)
    maj_top1, maj_topk = _topk_eval(lambda s: marginal_rank, test_min)
    print(f"{'majority-class':22s} top1={maj_top1:.3f} top{TOPK}={maj_topk:.3f}")

    # Order-1 Markov.
    markov_table = defaultdict(Counter)
    for sh in tr_shifts:
        acts = [action_key(s) for s in sh.steps]
        for prev, nxt in zip(acts, acts[1:]):
            markov_table[(prev,)][nxt] += 1

    def markov_pred(_state, _cache={}):
        # state for S1 is (location,), which equals the current action symbol
        ctx = (_state[0],)
        ctr = markov_table.get(ctx)
        return [a for a, _ in ctr.most_common()] if ctr else marginal_rank

    mk_top1, mk_topk = _topk_eval(markov_pred, test_min)
    print(f"{'markov(prev-loc)':22s} top1={mk_top1:.3f} top{TOPK}={mk_topk:.3f}")

    # Count-based BC per state variant.
    print("\ncount-based BC  P(a|state)  (unseen state -> backoff to majority):")
    count_scores = {}
    unseen_frac = {}
    for name, state_fn in STATE_VARIANTS.items():
        table = defaultdict(Counter)
        for s, a in build_pairs(tr_shifts, state_fn, action_key):
            table[s][a] += 1
        test_pairs = build_pairs(te_shifts, state_fn, action_key)
        n_unseen = sum(1 for s, _ in test_pairs if s not in table)
        unseen_frac[name] = round(n_unseen / max(1, len(test_pairs)), 3)

        def pred(s, _t=table):
            ctr = _t.get(s)
            return [a for a, _ in ctr.most_common()] if ctr else marginal_rank

        t1, tk = _topk_eval(pred, test_pairs)
        count_scores[name] = t1
        print(
            f"  {name:26s} top1={t1:.3f} top{TOPK}={tk:.3f}  "
            f"(unseen states: {unseen_frac[name]:.0%})"
        )

    # Optional neural BC.
    nn_top1 = None
    if use_bc:
        try:
            nn_top1 = _run_bc_grouped(tr_shifts, te_shifts, action_key)
            print(f"\nneural BC (S5, embeddings) top1={nn_top1:.3f}")
        except ImportError:
            print("\n[bc] flax/optax not installed; skipping neural BC.")

    best_count = max(count_scores.values())
    best_state = max(count_scores, key=count_scores.get)
    lift_over_markov = best_count - mk_top1
    lift_over_majority = best_count - maj_top1
    print("\n--- verdict ---")
    print(f"best count-BC state: {best_state}  top1={best_count:.3f}")
    print(
        f"lift over majority: {lift_over_majority:+.3f}   "
        f"lift over markov: {lift_over_markov:+.3f}"
    )
    if lift_over_majority < 0.03:
        print("RULE OUT: < 3 points over majority.")
    elif lift_over_markov < 0.02:
        print("PARTIAL: beats majority, < 2 points over order-1 Markov.")
    else:
        if nn_top1 is not None and nn_top1 - best_count > 0.02:
            print("KEEP: neural BC beats the count table.")
        else:
            print("KEEP: beats both baselines.")


def _run_bc_grouped(tr_shifts, te_shifts, action_key):
    """Neural BC with a grouped split (no shift shared across train/test)."""
    import jax
    import jax.numpy as jnp
    import optax
    from flax import linen as nn

    tr = build_pairs(tr_shifts, STATE_VARIANTS["S5_bigram"], action_key)
    te = build_pairs(te_shifts, STATE_VARIANTS["S5_bigram"], action_key)
    n_fields = len(tr[0][0])
    # Build vocab from TRAIN only; map unseen test values to a reserved index.
    vocabs = []
    for f in range(n_fields):
        vals = sorted({p[0][f] for p in tr}, key=str)
        vocabs.append({v: i for i, v in enumerate(vals)})  # unseen -> len(vocab)
    a_vals = sorted({a for _, a in tr}, key=str)
    a_vocab = {v: i for i, v in enumerate(a_vals)}

    def enc_x(pairs):
        return np.array(
            [
                [vocabs[f].get(p[0][f], len(vocabs[f])) for f in range(n_fields)]
                for p in pairs
            ]
        )

    Xtr, Xte = enc_x(tr), enc_x(te)
    ytr = np.array([a_vocab[a] for _, a in tr])
    # Test actions unseen in training count as misses.
    yte = np.array([a_vocab.get(a, -1) for _, a in te])

    class BC(nn.Module):
        sizes: tuple
        n_out: int

        @nn.compact
        def __call__(self, x):
            embs = [
                nn.Embed(self.sizes[f] + 1, 8)(x[:, f]) for f in range(len(self.sizes))
            ]
            h = jnp.concatenate(embs, axis=-1)
            h = nn.relu(nn.Dense(64)(h))
            return nn.Dense(self.n_out)(h)

    model = BC(tuple(len(v) for v in vocabs), len(a_vocab))
    params = model.init(jax.random.PRNGKey(0), jnp.array(Xtr[:2]))
    opt = optax.adam(1e-2)
    opt_state = opt.init(params)

    def loss_fn(p, xb, yb):
        logits = model.apply(p, xb)
        return optax.softmax_cross_entropy_with_integer_labels(logits, yb).mean()

    @jax.jit
    def step(p, os, xb, yb):
        loss, g = jax.value_and_grad(loss_fn)(p, xb, yb)
        u, os = opt.update(g, os)
        return optax.apply_updates(p, u), os, loss

    Xtrj, ytrj = jnp.array(Xtr), jnp.array(ytr)
    for _ in range(300):
        params, opt_state, _ = step(params, opt_state, Xtrj, ytrj)
    pred = np.array(model.apply(params, jnp.array(Xte)).argmax(-1))
    return float((pred == yte).mean())

### Revised rule-out and entropy

The Markov baseline is evaluated on action sequences, so `next_class` and `next_itype` are compared correctly; empty inputs raise an error.

In [ ]:
def conditional_entropy(pairs) -> tuple[float, float]:  # noqa: F811
    if not pairs:
        raise ValueError(
            "No state/action pairs were built. Check shift counts and MIN_SHIFT_EVENTS."
        )
    a_counts = Counter(a for _, a in pairs)
    n = sum(a_counts.values())
    H_a = -sum((c / n) * math.log2(c / n) for c in a_counts.values())
    by_state: defaultdict[tuple, Counter] = defaultdict(Counter)
    for s, a in pairs:
        by_state[s][a] += 1
    H_a_given_s = 0.0
    for ctr in by_state.values():
        m = sum(ctr.values())
        h = -sum((c / m) * math.log2(c / m) for c in ctr.values())
        H_a_given_s += (m / n) * h
    return H_a, H_a_given_s


def _action_transition_pairs(shift_list, action_key):
    pairs = []
    for sh in shift_list:
        actions = [action_key(step) for step in sh.steps]
        pairs.extend([((prev,), nxt) for prev, nxt in zip(actions, actions[1:])])
    return pairs


def run_ruleout(shifts, action_name="next_loc", use_bc=True):  # noqa: F811
    if len(shifts) < 2:
        raise ValueError("Need at least two shifts for a grouped train/test split.")
    action_key = ACTION_KEYS[action_name]
    tr_shifts, te_shifts = grouped_split(shifts)
    print(f"\n=== BC rule-out (action = {action_name}) ===")
    print(f"grouped split: {len(tr_shifts)} train shifts, {len(te_shifts)} test shifts")

    tr_actions = [action_key(s) for sh in tr_shifts for s in sh.steps]
    marginal_rank = [a for a, _ in Counter(tr_actions).most_common()]
    if not marginal_rank:
        raise ValueError("Training split contains no actions.")

    # Majority baseline, evaluated on action transitions.
    test_transitions = _action_transition_pairs(te_shifts, action_key)
    maj_top1, maj_topk = _topk_eval(lambda _ctx: marginal_rank, test_transitions)
    print(f"{'majority-class':22s} top1={maj_top1:.3f} top{TOPK}={maj_topk:.3f}")

    # Order-1 Markov on action sequences.
    markov_table = defaultdict(Counter)
    for ctx, nxt in _action_transition_pairs(tr_shifts, action_key):
        markov_table[ctx][nxt] += 1

    def markov_pred(ctx):
        ctr = markov_table.get(ctx)
        return [a for a, _ in ctr.most_common()] if ctr else marginal_rank

    mk_top1, mk_topk = _topk_eval(markov_pred, test_transitions)
    print(f"{'markov(prev-action)':22s} top1={mk_top1:.3f} top{TOPK}={mk_topk:.3f}")

    print("\ncount-based BC  P(a|state)  (unseen state -> backoff to majority):")
    count_scores = {}
    unseen_frac = {}
    for name, state_fn in STATE_VARIANTS.items():
        table = defaultdict(Counter)
        for state, action in build_pairs(tr_shifts, state_fn, action_key):
            table[state][action] += 1
        test_pairs = build_pairs(te_shifts, state_fn, action_key)
        n_unseen = sum(1 for state, _ in test_pairs if state not in table)
        unseen_frac[name] = round(n_unseen / max(1, len(test_pairs)), 3)

        def pred(state, _table=table):
            ctr = _table.get(state)
            return [a for a, _ in ctr.most_common()] if ctr else marginal_rank

        t1, tk = _topk_eval(pred, test_pairs)
        count_scores[name] = t1
        print(
            f"  {name:26s} top1={t1:.3f} top{TOPK}={tk:.3f}  "
            f"(unseen states: {unseen_frac[name]:.0%})"
        )

    nn_top1 = None
    if use_bc:
        try:
            nn_top1 = _run_bc_grouped(tr_shifts, te_shifts, action_key)
            print(f"\nneural BC (S5, embeddings) top1={nn_top1:.3f}")
        except ImportError:
            print("\n[bc] JAX/Flax/Optax not installed; skipping neural BC.")

    best_count = max(count_scores.values())
    best_state = max(count_scores, key=count_scores.get)
    lift_over_markov = best_count - mk_top1
    lift_over_majority = best_count - maj_top1
    print("\n--- verdict ---")
    print(f"best count-BC state: {best_state}  top1={best_count:.3f}")
    print(
        f"lift over majority: {lift_over_majority:+.3f}   lift over markov: {lift_over_markov:+.3f}"
    )
    if lift_over_majority < 0.03:
        print("RULE OUT: < 3 points over majority.")
    elif lift_over_markov < 0.02:
        print("PARTIAL: beats majority, < 2 points over order-1 Markov.")
    elif nn_top1 is not None and nn_top1 - best_count > 0.02:
        print("KEEP: neural BC beats the count table.")
    else:
        print("KEEP: beats both baselines.")

In [ ]:
def run_sweep(shifts, action_name="next_loc", use_bc=False):
    action_key = ACTION_KEYS[action_name]
    print(f"\n=== State-variant sweep (action = {action_name}) ===")
    print(f"{'variant':28s} {'top1':>6s} {'top3':>6s}")
    prev_top1 = None
    for name, state_fn in STATE_VARIANTS.items():
        pairs = build_pairs(shifts, state_fn, action_key)
        tr, te = split_pairs(pairs)
        # Reuse MarkovModel as a generic P(a|state) lookup over the full state.
        m = MarkovModel(order=0)
        m.fit(tr)
        r = m.evaluate(te)
        gain = "" if prev_top1 is None else f"  ({r['top1'] - prev_top1:+.3f})"
        print(f"{name:28s} {r['top1']:6.3f} {r[f'top{TOPK}']:6.3f}{gain}")
        prev_top1 = r["top1"]
    if use_bc:
        try:
            _run_bc(shifts, action_key)
        except ImportError:
            print("\n[bc] flax/optax not installed; skipping neural BC.")


def _run_bc(shifts, action_key):
    """Flax MLP BC over the S5 state."""
    import jax
    import jax.numpy as jnp
    import optax
    from flax import linen as nn

    pairs = build_pairs(shifts, STATE_VARIANTS["S5_bigram"], action_key)
    # Encode each categorical field with its own vocab.
    n_fields = len(pairs[0][0])
    vocabs = [
        {v: i for i, v in enumerate({p[0][f] for p in pairs})} for f in range(n_fields)
    ]
    a_vocab = {v: i for i, v in enumerate({a for _, a in pairs})}
    X = np.array([[vocabs[f][p[0][f]] for f in range(n_fields)] for p in pairs])
    y = np.array([a_vocab[a] for _, a in pairs])
    tr, te = split_pairs(list(range(len(y))))
    tr, te = np.array(tr), np.array(te)

    class BC(nn.Module):
        sizes: tuple
        n_out: int

        @nn.compact
        def __call__(self, x):
            embs = [nn.Embed(self.sizes[f], 8)(x[:, f]) for f in range(len(self.sizes))]
            h = jnp.concatenate(embs, axis=-1)
            h = nn.relu(nn.Dense(64)(h))
            return nn.Dense(self.n_out)(h)

    model = BC(tuple(len(v) for v in vocabs), len(a_vocab))
    key = jax.random.PRNGKey(0)
    params = model.init(key, jnp.array(X[:2]))
    opt = optax.adam(1e-2)
    opt_state = opt.init(params)

    def loss_fn(p, xb, yb):
        logits = model.apply(p, xb)
        return optax.softmax_cross_entropy_with_integer_labels(logits, yb).mean()

    @jax.jit
    def step(p, os, xb, yb):
        loss, g = jax.value_and_grad(loss_fn)(p, xb, yb)
        u, os = opt.update(g, os)
        return optax.apply_updates(p, u), os, loss

    Xtr, ytr = jnp.array(X[tr]), jnp.array(y[tr])
    for _ in range(300):
        params, opt_state, _ = step(params, opt_state, Xtr, ytr)
    logits = model.apply(params, jnp.array(X[te]))
    pred = np.array(logits.argmax(-1))
    top1 = float((pred == y[te]).mean())
    print(f"\n[bc] S5_bigram held-out top1 = {top1:.3f}")

## 10. Run analyses

In [ ]:
if RUN_BATTERY:
    if not shifts:
        raise RuntimeError("Load shifts first by setting RUN_LOAD_SHIFTS = True.")
    run_battery(shifts, ACTION_NAME, order=MARKOV_ORDER)
else:
    print("Battery skipped.")

In [ ]:
if RUN_RULEOUT:
    if not shifts:
        raise RuntimeError("Load shifts first by setting RUN_LOAD_SHIFTS = True.")
    run_ruleout(shifts, ACTION_NAME, use_bc=USE_NEURAL_BC)
else:
    print("Rule-out skipped.")

In [ ]:
if RUN_SWEEP:
    if not shifts:
        raise RuntimeError("Load shifts first by setting RUN_LOAD_SHIFTS = True.")
    run_sweep(shifts, ACTION_NAME, use_bc=USE_NEURAL_BC)
else:
    print("Sweep skipped.")

## 11. Log a stage to file

`capture_stage` writes a function's printed output to a log in `/outputs`.

In [ ]:
def capture_stage(log_name: str, fn: Callable, *args, **kwargs):
    buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(buffer), contextlib.redirect_stderr(buffer):
            result = fn(*args, **kwargs)
    except Exception:
        traceback.print_exc(file=buffer)
        result = None
        failed = True
    else:
        failed = False

    text = buffer.getvalue()
    print(text)
    path = OUTPUT_DIR / log_name
    path.write_text(text, encoding="utf-8")
    print("wrote:", path)
    if failed:
        raise RuntimeError(f"Stage failed; see {path}")
    return result


# Examples after shifts are loaded:
# capture_stage("battery.log", run_battery, shifts, "next_loc", 1)
# capture_stage("ruleout_class.log", run_ruleout, shifts, "next_class", False)

## 12. Close connection

In [ ]:
# con.close()
# print("connection closed")
